# Cleanup All Resources - GCP Dataplex Demo

This notebook will clean up and delete all resources created by `config_and_data_setup.ipynb`.

**Resources that will be deleted:**
1. BigQuery Dataset: `census_bureau_acs` (and all ~278 tables)
2. Dataplex Aspect Types:
   - `census-survey-metadata`
   - `data-governance-public`
3. All aspects applied to BigQuery table entries

**⚠️  WARNING:**
- This operation cannot be undone
- All data in the census_bureau_acs dataset will be permanently deleted
- All custom metadata (aspects) will be removed

**Required permissions:**
- `roles/bigquery.admin` (to delete BigQuery dataset)
- `roles/dataplex.admin` (to delete aspect types)
- `roles/dataplex.catalogAdmin` (to remove aspects from entries)

## Section 1: Configuration & Authentication

In [9]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
CATALOG_LOCATION = "us"  # BigQuery US multi-region datasets are cataloged in 'us'
DATASET_ID = "census_bureau_acs"
DATASET_LOCATION = "US"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Catalog Location: {CATALOG_LOCATION}")
print(f"  Dataset to delete: {DATASET_ID}")

📋 Configuration:
  Project ID: johnswain-1200-20260303091357
  Region: us-central1
  Catalog Location: us
  Dataset to delete: census_bureau_acs


In [10]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

🔐 Authentication Status:
  ✅ Authenticated successfully
  ✅ Default project: graph-demo-471710
  ⚠️  Note: Using PROJECT_ID (johnswain-1200-20260303091357) instead of default (graph-demo-471710)


In [11]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = ["google-cloud-bigquery", "google-cloud-dataplex"]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

📦 Installing required libraries...



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ Libraries installed



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
# Initialize clients
from google.cloud import bigquery
from google.cloud import dataplex_v1

bq_client = bigquery.Client(project=PROJECT_ID)
catalog_client = dataplex_v1.CatalogServiceClient()

print("✅ Clients initialized:")
print("  - BigQuery client")
print("  - Dataplex Catalog client")

✅ Clients initialized:
  - BigQuery client
  - Dataplex Catalog client


## Section 2: Remove Aspects from BigQuery Table Entries

Before deleting the aspect types, we need to remove all aspect instances that were applied to BigQuery table entries.

This will:
- List all tables in the census_bureau_acs dataset
- Remove both aspects (census-survey-metadata and data-governance-public) from each table entry

In [13]:
# List all tables in the dataset
print("📋 Listing tables in dataset...")
print()

dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"

try:
    tables = list(bq_client.list_tables(dataset_ref))
    total_tables = len(tables)
    
    print(f"✅ Found {total_tables} tables in {DATASET_ID}")
    print(f"   Will remove aspects from all {total_tables} table entries")
    print()
    
except Exception as e:
    error_msg = str(e)
    if "404" in error_msg or "not found" in error_msg.lower():
        print(f"⚠️  Dataset {DATASET_ID} not found - skipping aspect removal")
        total_tables = 0
    else:
        print(f"❌ Error listing tables: {error_msg}")
        raise

📋 Listing tables in dataset...

✅ Found 278 tables in census_bureau_acs
   Will remove aspects from all 278 table entries



In [14]:
# Remove aspects from all table entries
print("🗑️  Removing aspects from BigQuery table entries...")
print()

if total_tables == 0:
    print("⚠️  No tables found - skipping aspect removal")
else:
    # Aspect IDs to remove
    aspect_ids = ["census-survey-metadata", "data-governance-public"]
    
    catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"
    BQ_ENTRY_GROUP = "@bigquery"
    
    success_count = 0
    not_found_count = 0
    error_count = 0
    errors = []
    
    print(f"⏳ Processing {total_tables} tables...")
    
    for i, table in enumerate(tables):
        table_id = table.table_id
        
        # Construct the Dataplex entry name
        entry_id = f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
        entry_name = f"{catalog_parent}/entryGroups/{BQ_ENTRY_GROUP}/entries/{entry_id}"
        
        # Try to delete both aspects
        for aspect_id in aspect_ids:
            aspect_key = f"{PROJECT_ID}.{CATALOG_LOCATION}.{aspect_id}"
            aspect_full_name = f"{entry_name}/aspects/{aspect_key}"
            
            try:
                catalog_client.delete_aspect(name=aspect_full_name)
                success_count += 1
                
            except Exception as e:
                error_msg = str(e)
                
                # Not found is expected if aspect wasn't applied
                if "404" in error_msg or "not found" in error_msg.lower():
                    not_found_count += 1
                else:
                    error_count += 1
                    errors.append((table_id, aspect_id, error_msg[:80]))
        
        # Show progress every 25 tables
        if (i + 1) % 25 == 0:
            print(f"   Processed {i + 1}/{total_tables} tables... ({success_count} aspects removed)")
    
    print()
    print("=" * 60)
    print("✅ Aspect removal completed!")
    print("=" * 60)
    print(f"   Tables processed: {total_tables}")
    print(f"   Aspects removed: {success_count}")
    print(f"   Not found (already removed): {not_found_count}")
    print(f"   Errors: {error_count}")
    
    if error_count > 0:
        print()
        print(f"⚠️  Errors encountered ({error_count}):")
        for table_id, aspect_id, error in errors[:5]:
            print(f"   - {table_id}.{aspect_id}: {error}")
        if len(errors) > 5:
            print(f"   ... and {len(errors) - 5} more errors")

🗑️  Removing aspects from BigQuery table entries...

⏳ Processing 278 tables...
   Processed 25/278 tables... (0 aspects removed)
   Processed 50/278 tables... (0 aspects removed)
   Processed 75/278 tables... (0 aspects removed)
   Processed 100/278 tables... (0 aspects removed)
   Processed 125/278 tables... (0 aspects removed)
   Processed 150/278 tables... (0 aspects removed)
   Processed 175/278 tables... (0 aspects removed)
   Processed 200/278 tables... (0 aspects removed)
   Processed 225/278 tables... (0 aspects removed)
   Processed 250/278 tables... (0 aspects removed)
   Processed 275/278 tables... (0 aspects removed)

✅ Aspect removal completed!
   Tables processed: 278
   Aspects removed: 0
   Not found (already removed): 0
   Errors: 556

⚠️  Errors encountered (556):
   - blockgroup_2010_5yr.census-survey-metadata: 'CatalogServiceClient' object has no attribute 'delete_aspect'
   - blockgroup_2010_5yr.data-governance-public: 'CatalogServiceClient' object has no attribut

## Section 3: Delete Dataplex Aspect Types

Now we'll delete the custom aspect types created for the demo:
- `census-survey-metadata`
- `data-governance-public`

In [15]:
# Delete aspect types
print("🗑️  Deleting Dataplex Aspect Types...")
print()

aspect_type_ids = ["census-survey-metadata", "data-governance-public"]
catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"

deleted_count = 0
not_found_count = 0
error_count = 0

for aspect_type_id in aspect_type_ids:
    aspect_type_name = f"{catalog_parent}/aspectTypes/{aspect_type_id}"
    
    print(f"🗑️  Deleting: {aspect_type_id}")
    
    try:
        # Delete the aspect type
        delete_operation = catalog_client.delete_aspect_type(name=aspect_type_name)
        
        # Wait for deletion to complete
        delete_operation.result(timeout=300)
        
        print(f"   ✅ Deleted: {aspect_type_id}")
        deleted_count += 1
        
    except Exception as e:
        error_msg = str(e)
        
        if "404" in error_msg or "not found" in error_msg.lower():
            print(f"   ⚠️  Not found: {aspect_type_id} (already deleted)")
            not_found_count += 1
        else:
            print(f"   ❌ Error deleting {aspect_type_id}: {error_msg[:80]}")
            error_count += 1
    
    print()

print("=" * 60)
print("✅ Aspect Type deletion completed!")
print("=" * 60)
print(f"   Deleted: {deleted_count}")
print(f"   Not found: {not_found_count}")
print(f"   Errors: {error_count}")

🗑️  Deleting Dataplex Aspect Types...

🗑️  Deleting: census-survey-metadata
   ✅ Deleted: census-survey-metadata

🗑️  Deleting: data-governance-public
   ⚠️  Not found: data-governance-public (already deleted)

✅ Aspect Type deletion completed!
   Deleted: 1
   Not found: 1
   Errors: 0


## Section 4: Delete BigQuery Dataset

Finally, we'll delete the entire BigQuery dataset, which includes all ~278 census tables.

**⚠️  WARNING:** This will permanently delete all data in the `census_bureau_acs` dataset.

In [16]:
# Delete BigQuery dataset
print("🗑️  Deleting BigQuery Dataset...")
print()

dataset_id = f"{PROJECT_ID}.{DATASET_ID}"

print(f"⚠️  About to delete dataset: {dataset_id}")
print(f"   This will delete all {total_tables} tables and their data")
print()

try:
    # Delete the dataset with delete_contents=True to delete all tables
    bq_client.delete_dataset(
        dataset_id,
        delete_contents=True,  # Delete all tables in the dataset
        not_found_ok=True      # Don't error if already deleted
    )
    
    print("=" * 60)
    print("✅ BigQuery dataset deleted successfully!")
    print("=" * 60)
    print(f"   Dataset: {DATASET_ID}")
    print(f"   All {total_tables} tables and data have been permanently deleted")
    
except Exception as e:
    error_msg = str(e)
    
    if "404" in error_msg or "not found" in error_msg.lower():
        print("⚠️  Dataset not found (already deleted)")
    else:
        print(f"❌ Error deleting dataset: {error_msg}")
        raise

🗑️  Deleting BigQuery Dataset...

⚠️  About to delete dataset: johnswain-1200-20260303091357.census_bureau_acs
   This will delete all 278 tables and their data

✅ BigQuery dataset deleted successfully!
   Dataset: census_bureau_acs
   All 278 tables and data have been permanently deleted


## Section 5: Cleanup Summary

Review the cleanup results.

In [ ]:
# Summary
print()
print("=" * 60)
print("🎉 CLEANUP COMPLETED!")
print("=" * 60)
print()
print("📊 Summary of deleted resources:")
print()
print("   1. BigQuery Dataset:")
print(f"      - Dataset: {DATASET_ID}")
print(f"      - Tables deleted: {total_tables}")
print()
print("   2. Dataplex Aspect Types:")
print("      - census-survey-metadata")
print("      - data-governance-public")
print()
print("   3. Dataplex Aspects:")
print(f"      - Removed from {total_tables} table entries")
print()
print("✅ All demo resources have been cleaned up!")
print()
print("🔗 Verify in Console:")
print(f"   BigQuery Datasets: https://console.cloud.google.com/bigquery?project={PROJECT_ID}")
print(f"   Dataplex Aspect Types: https://console.cloud.google.com/dataplex/govern/aspect-types?project={PROJECT_ID}")
print(f"   Dataplex Catalog: https://console.cloud.google.com/dataplex/search?project={PROJECT_ID}")

## Optional: Clean up IAM permissions

If you want to remove the IAM permissions that were granted for the demo, run these commands in your terminal:

```bash
# Remove BigQuery Admin
gcloud projects remove-iam-policy-binding johnswain-1200-20260303091357 \
  --member='user:YOUR_EMAIL' \
  --role='roles/bigquery.admin'

# Remove Dataplex Admin
gcloud projects remove-iam-policy-binding johnswain-1200-20260303091357 \
  --member='user:YOUR_EMAIL' \
  --role='roles/dataplex.admin'

# Remove Dataplex Catalog Admin
gcloud projects remove-iam-policy-binding johnswain-1200-20260303091357 \
  --member='user:YOUR_EMAIL' \
  --role='roles/dataplex.catalogAdmin'
```

Replace `YOUR_EMAIL` with your Google Cloud account email.